# TripMe - Generate Historical Significance Field (Wikipedia + LLM summarization)

For every place record under `data/raw/<District>/<category>.jsonl`, looks up
a real Wikipedia article for that place, and if one is found, asks the LLM
(Qwen2.5-3B-Instruct) to summarize ONLY the facts in that article into a
~60-100 word `historical_significance` field. The LLM never answers from its
own memory here - every claim has to be traceable to the fetched Wikipedia
text, which is what keeps this grounded instead of hallucinated. If no
matching Wikipedia article is found, no field is added (nothing invented).

This is what feeds the "oracle" chat scenario: a user asking "what's the
historical significance of Ruwanwelisaya?" should get a real, sourced
answer, and the training data should only include that kind of Q&A for
places we actually found a source for.

## How to run this on Kaggle
1. Zip your local `data/raw` folder and upload it as a Kaggle dataset (e.g. named
   `tripme-raw-places`), then attach it to this notebook via "Add Input".
2. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or better).
3. **Turn on Internet** (Settings -> Internet -> On) - this notebook calls the
   live Wikipedia API, unlike the description-regeneration notebook which is
   fully offline.
4. Run all cells. Output files land in `/kaggle/working/raw_updated_history/` -
   download that folder (Kaggle auto-zips the working directory) and copy its
   contents over `data/raw` locally (merges the new field into existing records).

This notebook is resumable: progress is checkpointed to
`/kaggle/working/history_progress.json`, so if the Kaggle session gets interrupted
partway through, re-running continues where it left off instead of starting over.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece tqdm requests
print("Dependencies installed.")

In [ ]:
import json
import time
from pathlib import Path

import requests
from tqdm.auto import tqdm

# Where the uploaded data/raw folder is mounted. Kaggle dataset slugs don't
# always match what you typed and the folder structure inside the zip can
# vary, so search for it instead of hardcoding one exact path.
RAW_INPUT_DIR = None
_kaggle_input = Path("/kaggle/input")
if _kaggle_input.exists():
    _candidates = [p for p in _kaggle_input.rglob("*") if p.is_dir() and p.name == "raw"]
    if not _candidates:
        _candidates = [
            p for p in _kaggle_input.iterdir()
            if p.is_dir() and any(p.rglob("*.jsonl"))
        ]
    if _candidates:
        RAW_INPUT_DIR = _candidates[0]
        print(f"Auto-detected input dataset folder: {RAW_INPUT_DIR}")
    else:
        print("WARNING: /kaggle/input exists but no folder named 'raw' (or "
              "containing .jsonl files) was found inside it. Check the Input "
              "panel on the right and set RAW_INPUT_DIR manually below.")

if RAW_INPUT_DIR is None:
    RAW_INPUT_DIR = Path("data/raw")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
RAW_OUTPUT_DIR = OUTPUT_DIR / "raw_updated_history"
PROGRESS_FILE = OUTPUT_DIR / "history_progress.json"

RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading from:", RAW_INPUT_DIR)
print("Writing to:", RAW_OUTPUT_DIR)

In [ ]:
# Quick check that Wikipedia is reachable - if this fails, go to
# Settings -> Internet -> On and re-run.
_wiki_check_session = requests.Session()
_wiki_check_session.headers.update({
    "User-Agent": "TripMe-SriLanka-DataPipeline/1.0 (educational travel-app dataset project)"
})
try:
    _r = _wiki_check_session.get("https://en.wikipedia.org/w/api.php", params={"action": "query", "format": "json", "meta": "siteinfo"}, timeout=10)
    _r.raise_for_status()
    print("Wikipedia API reachable.")
except Exception as e:
    raise SystemExit(
        f"Could not reach Wikipedia ({e}). On Kaggle, go to Settings -> "
        "Internet -> On (right sidebar) and re-run this notebook."
    )

## Load every place record from data/raw

In [ ]:
def load_records(path: Path):
    text = path.read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)
    while idx < n:
        while idx < n and text[idx] in " \t\r\n":
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(text, idx)
        yield obj
        idx = end


all_files = sorted(RAW_INPUT_DIR.rglob("*.jsonl"))
print(f"Found {len(all_files)} .jsonl files under {RAW_INPUT_DIR}")

records_by_file = {}
total_records = 0
for f in all_files:
    recs = list(load_records(f))
    records_by_file[f] = recs
    total_records += len(recs)

print(f"Loaded {total_records} place records total.")

## Wikipedia lookup

Two-step: (1) search for the place name (plus "Sri Lanka" to disambiguate
from same-named places elsewhere in the world), take the top hit; (2) fetch
that page's plain-text extract. Both use MediaWiki's public API - no key
needed, but be a considerate API citizen with a real User-Agent and a small
delay between requests.

In [ ]:
WIKI_SESSION = requests.Session()
WIKI_SESSION.headers.update({
    "User-Agent": "TripMe-SriLanka-DataPipeline/1.0 (educational travel-app dataset project)"
})
WIKI_REQUEST_DELAY_SEC = 0.2  # be polite to the free API

# Wikipedia extracts often contain a trailing disambiguation/stub boilerplate
# or are themselves disambiguation pages - skip anything that's clearly not
# a real substantive article about the place.
MIN_EXTRACT_CHARS = 200


def wikipedia_search_title(query: str) -> str | None:
    try:
        resp = WIKI_SESSION.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "list": "search",
                "srsearch": query,
                "srlimit": 1,
                "format": "json",
            },
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get("query", {}).get("search", [])
        return results[0]["title"] if results else None
    except Exception:
        return None


def wikipedia_fetch_extract(title: str) -> str | None:
    try:
        resp = WIKI_SESSION.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "prop": "extracts",
                "explaintext": 1,
                "exintro": 0,
                "titles": title,
                "format": "json",
            },
            timeout=15,
        )
        resp.raise_for_status()
        pages = resp.json().get("query", {}).get("pages", {})
        for page in pages.values():
            extract = page.get("extract", "")
            if extract and "may refer to" not in extract[:200].lower():
                return extract
        return None
    except Exception:
        return None


def find_wikipedia_article(rec: dict) -> tuple[str, str] | None:
    """Returns (title, extract_text) for the best-matching Wikipedia article,
    or None if nothing substantive was found."""
    name = rec.get("name", "").strip()
    district = rec.get("district_id", "").strip()
    if not name:
        return None

    for query in (f"{name} {district} Sri Lanka", f"{name} Sri Lanka"):
        title = wikipedia_search_title(query)
        time.sleep(WIKI_REQUEST_DELAY_SEC)
        if not title:
            continue
        extract = wikipedia_fetch_extract(title)
        time.sleep(WIKI_REQUEST_DELAY_SEC)
        if extract and len(extract) >= MIN_EXTRACT_CHARS:
            return title, extract
    return None


# Smoke test - a well-documented site should find a real article; an
# obscure/made-up name should find nothing.
print("Ruwanwelisaya lookup:", (lambda r: (r[0], len(r[1])) if r else None)(
    find_wikipedia_article({"name": "Ruwanwelisaya", "district_id": "Anuradhapura"})
))
print("Nonsense lookup:", find_wikipedia_article({"name": "Zzqx Made Up Falls Xyzabc", "district_id": "Mullaitivu"}))

## Load the LLM (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

llm_ready = False
llm_tokenizer = None
llm_model = None

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if torch.cuda.is_available():
        print(f"Loading {LLM_MODEL_NAME} (4-bit)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        llm_model.eval()
        llm_ready = True
        print("LLM loaded and ready.")
    else:
        print("No GPU detected. This notebook needs a GPU - enable one via "
              "Settings -> Accelerator and re-run.")
        raise SystemExit("No GPU available.")
except ImportError as e:
    raise SystemExit(f"Missing dependency ({e}) - re-run the pip install cell above.")

## Summarize the fetched Wikipedia text (grounded - LLM may only use the given article text)

In [ ]:
HISTORY_SYSTEM_PROMPT = (
    "You summarize Wikipedia article text into a short historical/cultural "
    "significance blurb for a Sri Lanka travel app. You must use ONLY the "
    "facts present in the article text you are given below - do not add any "
    "fact, date, name, or claim that is not stated in that text, even if you "
    "think you know more about the place from elsewhere. If the article text "
    "given to you does not actually contain any historical or cultural "
    "significance (e.g. it's just geographic/administrative facts), reply "
    "with exactly the single token NO_HISTORY_IN_ARTICLE and nothing else. "
    "Otherwise, write 60-100 words, one paragraph, no markdown, no preamble, "
    "no meta-commentary about the source - just the historical/cultural "
    "content itself, in your own words."
)

NO_HISTORY_MARKER = "NO_HISTORY_IN_ARTICLE"

# Article extracts can be long (multiple sections); truncate to keep the
# prompt a reasonable size for a 3B model - the lead + early sections
# usually carry the historical significance anyway.
MAX_ARTICLE_CHARS = 3000


def build_history_prompt(rec: dict, article_title: str, article_text: str) -> str:
    name = rec.get("name", "")
    truncated = article_text[:MAX_ARTICLE_CHARS]
    return (
        f'Place name: "{name}"\n'
        f'Wikipedia article: "{article_title}"\n\n'
        f"Article text:\n{truncated}\n\n"
        "Summarize this article's historical/cultural significance in "
        "60-100 words, using only facts stated above. If there is no "
        "historical/cultural content in this text, reply with exactly "
        f"{NO_HISTORY_MARKER}."
    )


BATCH_SIZE = 16  # smaller than the description notebook - prompts are much longer here (full article text)

llm_tokenizer.padding_side = "left"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token


def llm_generate_batch(prompts: list[str], max_new_tokens=170, temperature=0.2) -> list[str]:
    # Low temperature: this is grounded summarization, not creative writing -
    # we want the model to stick closely to the source text.
    formatted = [
        llm_tokenizer.apply_chat_template(
            [{"role": "system", "content": HISTORY_SYSTEM_PROMPT},
             {"role": "user", "content": p}],
            add_generation_prompt=True, tokenize=False,
        )
        for p in prompts
    ]
    inputs = llm_tokenizer(formatted, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    texts = llm_tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)
    return [t.strip() for t in texts]


def is_valid_history(text: str) -> bool:
    if not text:
        return False
    word_count = len(text.split())
    return 40 <= word_count <= 130


def summarize_articles_batch(items: list[tuple[dict, str, str]], max_retries=2) -> dict:
    """items: list of (record, article_title, article_text). Returns
    {id: historical_significance_or_None}. None means the article had no
    real historical content, or generation kept failing validation."""
    results = {}
    remaining = items
    for attempt in range(max_retries + 1):
        if not remaining:
            break
        prompts = [build_history_prompt(rec, title, text) for rec, title, text in remaining]
        try:
            texts = llm_generate_batch(prompts)
        except Exception as e:
            print(f"  batch generation error (attempt {attempt+1}): {e}")
            continue
        still_remaining = []
        for (rec, title, atext), out_text in zip(remaining, texts):
            if NO_HISTORY_MARKER in out_text:
                results[rec["id"]] = None
            elif is_valid_history(out_text):
                results[rec["id"]] = out_text
            else:
                still_remaining.append((rec, title, atext))
        remaining = still_remaining
    for rec, _, _ in remaining:
        results[rec["id"]] = None
    return results

## Process every record: Wikipedia lookup, then batched LLM summarization

Lookup is sequential (network-bound, one place at a time), but once a batch
of articles is collected, summarization runs as a single batched GPU call -
same throughput trick as the description-regeneration notebook.

In [ ]:
MAX_RUNTIME_HOURS = 9.0
run_deadline = time.time() + MAX_RUNTIME_HOURS * 3600


def load_progress() -> dict:
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {"done": {}}  # id -> historical_significance text, or null if none found


def save_progress(progress: dict) -> None:
    PROGRESS_FILE.write_text(json.dumps(progress), encoding="utf-8")


progress = load_progress()
done_map = progress["done"]
print(f"Resuming: {len(done_map)} places already processed in a previous run.")

all_records = [rec for recs in records_by_file.values() for rec in recs]
pending = [rec for rec in all_records if rec.get("id") not in done_map]
already_have_history = sum(1 for v in done_map.values() if v)
print(f"{already_have_history} places already have a real historical_significance.")
print(f"{len(done_map) - already_have_history} places already checked, no article/history found.")
print(f"{len(pending)} of {len(all_records)} records still need processing.")
print(f"Will stop (and save progress) after {MAX_RUNTIME_HOURS} hours even if not "
      "finished - just re-run this notebook to continue.")

stopped_early = False
article_batch = []  # list of (record, title, extract) waiting to be summarized

def flush_batch():
    global done_map
    if not article_batch:
        return
    results = summarize_articles_batch(list(article_batch))
    done_map.update(results)
    article_batch.clear()
    save_progress(progress)

for rec in tqdm(pending, desc="Looking up Wikipedia + summarizing"):
    if time.time() >= run_deadline:
        stopped_early = True
        print(f"\nHit the {MAX_RUNTIME_HOURS}-hour safety limit - stopping and saving progress.")
        break

    found = find_wikipedia_article(rec)
    if found is None:
        done_map[rec["id"]] = None
        continue

    title, extract = found
    article_batch.append((rec, title, extract))
    if len(article_batch) >= BATCH_SIZE:
        flush_batch()

flush_batch()  # summarize any leftover partial batch
save_progress(progress)

found_count = sum(1 for v in done_map.values() if v)
print(f"\nDone. {len(done_map)} places processed ({found_count} with real sourced history, "
      f"{len(done_map) - found_count} with no article or no historical content found).")
print(f"{len(all_records) - len(done_map)} still pending"
      + (" - stopped early on the time limit, re-run to continue." if stopped_early
         else " - re-run this cell to retry any that errored out."))

## Write updated records back out, same per-district/per-category layout

Only records where a Wikipedia article was found AND it contained real
historical/cultural content get the new `historical_significance` field.
Everything else is written out unchanged - this is intentional: the oracle
chat scenario should only be trained to answer history questions for
places we can point to a real source for.

In [ ]:
added_count = 0
unchanged_count = 0

for src_path, recs in records_by_file.items():
    rel = src_path.relative_to(RAW_INPUT_DIR)
    out_path = RAW_OUTPUT_DIR / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in recs:
            rid = rec.get("id")
            history = done_map.get(rid)
            if history:
                rec = dict(rec)
                rec["historical_significance"] = history
                added_count += 1
            else:
                unchanged_count += 1
            f.write(json.dumps(rec, indent=4, ensure_ascii=False))
            f.write("\n\n")

print(f"Wrote {len(records_by_file)} files to {RAW_OUTPUT_DIR}")
print(f"{added_count} records got a real, Wikipedia-sourced historical_significance field.")
print(f"{unchanged_count} records had no matching article or no historical content - left unchanged.")
print("\nNext step: download /kaggle/working (Kaggle auto-zips it), then copy the "
      "contents of raw_updated_history/ over your local data/raw folder (this "
      "merges the new field into your existing records; run 01_merge_places.py "
      "afterward to refresh data/processed/places.json).")